# 🤖 ClipNest — Telegram Bot on Google Colab

Is notebook se **Telegram downloader bot** deploy hoga. Web server / Cloudflare tunnel start nahi hoga.

### Start karne ka tarika
1. `Runtime → Run all` karein.
2. BotFather ka token paste karein.
3. Bot ready hone par Telegram mein `/start` bhejein.

⚠️ Colab temporary hai. Runtime disconnect/stop hote hi bot offline ho jayega. Token ko public notebook ya GitHub par save na karein. Sirf wahi content download karein jiska aapko haq hai.


## 1️⃣ Install dependencies + clone repository


In [ ]:
import subprocess, sys
from pathlib import Path

REPO = "https://github.com/gintokisakata918-prog/Tg-downloder-.git"
APP_DIR = Path("/content/Tg-downloder-")

def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(cmd, cwd=cwd, check=True)

# FFmpeg is used for merging video/audio and MP3 conversion.
subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True, check=True)

if APP_DIR.exists():
    subprocess.run(["git", "fetch", "origin", "main"], cwd=APP_DIR, check=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=APP_DIR, check=True)
else:
    run(["git", "clone", "--depth", "1", REPO, str(APP_DIR)])

# Install project and Telegram bot optional dependency.
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[bot]"] , cwd=APP_DIR)
(APP_DIR / "downloads").mkdir(exist_ok=True)
print("\n✅ Installation complete:", APP_DIR)


## 2️⃣ Bot settings

BotFather se token lein. Allowed user IDs blank chhodne par bot sabhi users ke liye open rahega. Private bot ke liye apni Telegram ID daalein (multiple IDs comma-separated).


In [ ]:
import os

BOT_TOKEN = input("BotFather BOT TOKEN paste karein: ").strip()
if not BOT_TOKEN or ":" not in BOT_TOKEN:
    raise ValueError("Valid Telegram bot token enter karein.")

ALLOWED_USERS = input("Allowed Telegram user IDs (optional, comma-separated; blank = everyone): ").strip()

os.environ["CLIPNEST_TELEGRAM_BOT_TOKEN"] = BOT_TOKEN
os.environ["CLIPNEST_TELEGRAM_ALLOWED_USERS"] = ALLOWED_USERS
os.environ["CLIPNEST_DOWNLOAD_DIR"] = "/content/Tg-downloder-/downloads"
os.environ["CLIPNEST_MAX_CONCURRENT_DOWNLOADS"] = "2"
os.environ["CLIPNEST_JOB_TTL_MINUTES"] = "30"
os.environ["CLIPNEST_MAX_FILESIZE_MB"] = "1024"
os.environ["CLIPNEST_MAX_DURATION_SECONDS"] = "7200"
os.environ["CLIPNEST_TELEGRAM_MAX_UPLOAD_MB"] = "50"
os.environ["CLIPNEST_RATE_LIMIT_PER_10MIN"] = "20"

print("✅ Settings ready. Token output mein print nahi kiya gaya.")


## 3️⃣ Start Telegram bot

Is cell ko run karne ke baad bot polling mode mein chalega. Cell running rehna zaroori hai. Telegram par bot open karke `/start` bhejein.

**Bot stop:** Colab ke toolbar mein ⏹ Stop button dabayein.


In [ ]:
import subprocess, sys

if not os.environ.get("CLIPNEST_TELEGRAM_BOT_TOKEN"):
    raise RuntimeError("Pehle settings cell run karein.")

print("🤖 Starting ClipNest Telegram bot...", flush=True)
print("Telegram mein bot ko /start bhejein. Stop karne ke liye cell ka Stop button dabayein.", flush=True)
subprocess.run([sys.executable, "-m", "app.telegram_bot"], cwd=APP_DIR, check=True)


## 🛠️ Troubleshooting

- **Conflict: terminated by other getUpdates request** — same token wala bot kisi aur Colab/runtime/server par bhi chal raha hai. Purana process band karein.
- **Unauthorized** — BotFather token galat hai; settings cell dobara run karein.
- **File too large** — Telegram Bot API par upload limit 50 MB hai; lower quality select karein.
- **Runtime disconnect** — Colab session temporary hai; reconnect karke cells dobara run karein.
- **Code update** — install/clone cell dobara run karein, phir settings aur start cells.

Bot commands: `/start`, `/help`, `/id`.
